In [16]:
import pandas as pd

df = pd.read_csv(r"C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\EMG\TimeFreq_200_100\Merged_Features\Merged_Features_TimeFreq_200_100.csv")
df.to_parquet(r"C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\EMG\TimeFreq_200_100\Merged_Features\Merged_Features_TimeFreq_200_100.parquet", index=False)

In [ ]:
import pandas as pd

df = pd.read_parquet(r"C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\EMG\TimeFreq_200_100\Merged_Features\Merged_Features_TimeFreq_200_100.parquet")
print(df.shape)
print(df.head())

(801315, 99)
  Participant     Sex  Age  Stature  Body Mass       Trial Number  \
0         P10  Female   26     1.73       71.4  Trial 1 - Fatigue   
1         P10  Female   26     1.73       71.4  Trial 1 - Fatigue   
2         P10  Female   26     1.73       71.4  Trial 1 - Fatigue   
3         P10  Female   26     1.73       71.4  Trial 1 - Fatigue   
4         P10  Female   26     1.73       71.4  Trial 1 - Fatigue   

           Activity  Lifting Height  Lowering Height  Box Mass  ...  \
0  Lifting-Lowering             0.0            0.265       6.8  ...   
1  Lifting-Lowering             0.0            0.265       6.8  ...   
2  Lifting-Lowering             0.0            0.265       6.8  ...   
3  Lifting-Lowering             0.0            0.265       6.8  ...   
4  Lifting-Lowering             0.0            0.265       6.8  ...   

  EMG_CH8_std  EMG_CH8_min  EMG_CH8_max  EMG_CH8_mav  EMG_CH8_rms  \
0  188.873074    47.783016   802.199633   648.474571   675.420097   
1   28.

In [ ]:
`# emg_rf_classifier_no_fatigue.py

import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
)
from sklearn.model_selection import GroupShuffleSplit


# =========================
# CONFIG
# =========================
INPUT_PARQUET = r"C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\EMG\TimeFreq_200_100\Merged_Features\Merged_Features_TimeFreq_200_100.parquet"
INPUT_CSV = r"C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\EMG\TimeFreq_200_100\Merged_Features\Merged_Features_TimeFreq_200_100.csv"
OUTPUT_DIR = r"C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\EMG\TimeFreq_200_100\Results"

RANDOM_STATE = 42
TEST_SIZE = 0.20

RF_PARAMS = {
    "n_estimators": 300,
    "max_depth": 10,
    "min_samples_split": 2,
    "min_samples_leaf": 3,
    "max_features": "sqrt",
    "bootstrap": True,
    "class_weight": "balanced_subsample",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

PARTICIPANT_CANDIDATES = ["Participant"]
SEX_CANDIDATES = ["Sex"]
TRIAL_NAME_CANDIDATES = ["Trial Number"]
TRIAL_ID_CANDIDATES = ["Timeline", "Timestamp", "Trial Number"]
TARGET_CANDIDATES = ["Box Weight", "Box Mass"]


# =========================
# HELPERS
# =========================
def ensure_dir(path):
    p = Path(path)
    p.mkdir(parents=True, exist_ok=True)
    return p


def resolve_column(df, candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise ValueError(f"None of these columns found: {candidates}")
    return ""


def normalize_sex(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s in {"m", "male"}:
        return "Male"
    if s in {"f", "female"}:
        return "Female"
    return str(x)


def build_trial_id(df, participant_col, trial_id_col):
    return (
        df[participant_col].astype(str).fillna("NA")
        + "||"
        + df[trial_id_col].astype(str).fillna("NA")
    )


def select_feature_columns(df, participant_col, sex_col, trial_id_col, target_col):
    excluded = {
        participant_col,
        sex_col,
        trial_id_col,
        target_col,
        "trial_id",
        "source_file",
        "start_time",
        "end_time",
        "window_start_idx",
        "window_end_idx",
        "window_size",
        "step_size",
        "Activity",
        "Load Type",
        "Lifting Height",
        "Lifting Depth",
        "Lowering Height",
        "Trial Number",
        "Age",
        "Stature",
        "Body Mass",
        "Timestamp",
        "Timeline",
        "Box Weight",
        "Box Mass",
    }

    feature_cols = [c for c in df.columns if c.startswith("EMG_CH") and c not in excluded]
    if not feature_cols:
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        feature_cols = [c for c in numeric_cols if c not in excluded]

    if not feature_cols:
        raise ValueError("No usable feature columns found.")

    return sorted(feature_cols)


def classification_metrics(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
    }


def numeric_error_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
    }


def aggregate_trial_probs(df_test, prob_cols, participant_col, sex_col, trial_id_col, target_col):
    agg_prob = df_test.groupby("trial_id", as_index=False)[prob_cols].mean()

    meta = (
        df_test.groupby("trial_id", as_index=False)
        .agg({
            participant_col: "first",
            sex_col: "first",
            trial_id_col: "first",
            target_col: "first",
        })
    )

    out = meta.merge(agg_prob, on="trial_id", how="left")
    return out


# =========================
# MAIN
# =========================
def main():
    out_dir = ensure_dir(OUTPUT_DIR)

    df = pd.read_parquet(INPUT_PARQUET)

    participant_col = resolve_column(df, PARTICIPANT_CANDIDATES, required=True)
    sex_col = resolve_column(df, SEX_CANDIDATES, required=True)
    trial_name_col = resolve_column(df, TRIAL_NAME_CANDIDATES, required=False)
    trial_id_col = resolve_column(df, TRIAL_ID_CANDIDATES, required=True)
    target_col = resolve_column(df, TARGET_CANDIDATES, required=True)

    df[sex_col] = df[sex_col].map(normalize_sex)

    # -------------------------
    # remove fatigue rows
    # -------------------------
    n_before = len(df)
    if trial_name_col:
        fatigue_mask = df[trial_name_col].astype(str).str.contains("fatigue", case=False, na=False)
        df = df.loc[~fatigue_mask].copy()
    n_after = len(df)

    df["trial_id"] = build_trial_id(df, participant_col, trial_id_col)

    feature_cols = select_feature_columns(df, participant_col, sex_col, trial_id_col, target_col)

    keep_cols = [participant_col, sex_col, trial_id_col, target_col, "trial_id"] + feature_cols
    optional_cols = [
        c for c in [
            "Trial Number", "Activity", "Lifting Height", "Lowering Height",
            "Load Type", "Lifting Depth", "Age", "Stature", "Body Mass",
            "window_start_idx", "window_end_idx", "window_size", "step_size", "source_file"
        ]
        if c in df.columns and c not in keep_cols
    ]
    work_df = df[keep_cols + optional_cols].copy()

    work_df = work_df.dropna(subset=[participant_col, sex_col, trial_id_col, target_col]).copy()
    X_all = work_df[feature_cols].replace([np.inf, -np.inf], np.nan)
    valid_mask = X_all.notna().all(axis=1)
    work_df = work_df.loc[valid_mask].reset_index(drop=True)

    if len(work_df) == 0:
        raise ValueError("No valid rows remain after filtering.")

    # sorted discrete class levels
    class_values = sorted(pd.to_numeric(work_df[target_col], errors="coerce").dropna().unique().tolist())
    class_to_idx = {v: i for i, v in enumerate(class_values)}
    idx_to_class = {i: v for v, i in class_to_idx.items()}

    work_df["y_class"] = work_df[target_col].map(class_to_idx)
    if work_df["y_class"].isna().any():
        raise ValueError("Some target values could not be mapped to classes.")

    # subject-level split preferred
    unique_participants = work_df[participant_col].dropna().astype(str).unique()
    if len(unique_participants) >= 2:
        split_groups = work_df[participant_col].astype(str).values
        split_strategy = "subject_level"
    else:
        split_groups = work_df["trial_id"].astype(str).values
        split_strategy = "trial_group_level"

    gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    train_idx, test_idx = next(gss.split(work_df, groups=split_groups))

    train_df = work_df.iloc[train_idx].reset_index(drop=True)
    test_df = work_df.iloc[test_idx].reset_index(drop=True)

    # safety check
    shared_trials = set(train_df["trial_id"]).intersection(set(test_df["trial_id"]))
    if len(shared_trials) > 0:
        raise RuntimeError("Leakage detected: some trials appear in both train and test.")

    X_train = train_df[feature_cols].values
    y_train = train_df["y_class"].astype(int).values

    X_test = test_df[feature_cols].values
    y_test = test_df["y_class"].astype(int).values

    clf = RandomForestClassifier(**RF_PARAMS)
    clf.fit(X_train, y_train)

    # -------------------------
    # window-level predictions
    # -------------------------
    test_df = test_df.copy()
    test_df["y_true_class"] = y_test
    test_df["y_pred_class"] = clf.predict(X_test)

    test_df["y_true_weight"] = test_df["y_true_class"].map(idx_to_class)
    test_df["y_pred_weight"] = test_df["y_pred_class"].map(idx_to_class)

    prob = clf.predict_proba(X_test)
    prob_cols = [f"prob_{idx_to_class[i]}" for i in range(len(class_values))]
    prob_df = pd.DataFrame(prob, columns=prob_cols, index=test_df.index)
    test_df = pd.concat([test_df, prob_df], axis=1)

    window_cls_metrics = classification_metrics(test_df["y_true_class"], test_df["y_pred_class"])
    window_num_metrics = numeric_error_metrics(test_df["y_true_weight"], test_df["y_pred_weight"])

    # -------------------------
    # trial-level aggregation
    # average probabilities over windows
    # -------------------------
    trial_df = aggregate_trial_probs(
        df_test=test_df,
        prob_cols=prob_cols,
        participant_col=participant_col,
        sex_col=sex_col,
        trial_id_col=trial_id_col,
        target_col=target_col,
    )

    prob_mat = trial_df[prob_cols].values
    trial_pred_idx = np.argmax(prob_mat, axis=1)
    trial_df["y_pred_class"] = trial_pred_idx
    trial_df["y_pred_weight"] = trial_df["y_pred_class"].map(idx_to_class)

    trial_df["y_true_weight"] = pd.to_numeric(trial_df[target_col], errors="coerce")
    trial_df["y_true_class"] = trial_df["y_true_weight"].map(class_to_idx)

    trial_cls_metrics = classification_metrics(trial_df["y_true_class"], trial_df["y_pred_class"])
    trial_num_metrics = numeric_error_metrics(trial_df["y_true_weight"], trial_df["y_pred_weight"])

    # counts
    train_trial_counts_per_weight = (
        train_df[[target_col, "trial_id"]]
        .drop_duplicates()
        .groupby(target_col, as_index=False)
        .size()
        .rename(columns={"size": "n_train_trials"})
        .sort_values(target_col)
        .reset_index(drop=True)
    )

    test_trial_counts_per_weight = (
        test_df[[target_col, "trial_id"]]
        .drop_duplicates()
        .groupby(target_col, as_index=False)
        .size()
        .rename(columns={"size": "n_test_trials"})
        .sort_values(target_col)
        .reset_index(drop=True)
    )

    feature_importance_df = pd.DataFrame({
        "feature": feature_cols,
        "importance": clf.feature_importances_,
    }).sort_values("importance", ascending=False).reset_index(drop=True)

    # confusion matrices
    cm_window = confusion_matrix(
        test_df["y_true_class"],
        test_df["y_pred_class"],
        labels=list(range(len(class_values)))
    )
    cm_trial = confusion_matrix(
        trial_df["y_true_class"],
        trial_df["y_pred_class"],
        labels=list(range(len(class_values)))
    )

    cm_window_df = pd.DataFrame(
        cm_window,
        index=[f"true_{v}" for v in class_values],
        columns=[f"pred_{v}" for v in class_values],
    )
    cm_trial_df = pd.DataFrame(
        cm_trial,
        index=[f"true_{v}" for v in class_values],
        columns=[f"pred_{v}" for v in class_values],
    )

    summary = {
        "input_csv": INPUT_CSV,
        "split_strategy": split_strategy,
        "n_rows_before_removing_fatigue": int(n_before),
        "n_rows_after_removing_fatigue": int(n_after),
        "n_removed_rows": int(n_before - n_after),
        "n_total_windows_used": int(len(work_df)),
        "n_train_windows": int(len(train_df)),
        "n_test_windows": int(len(test_df)),
        "n_total_trials_used": int(work_df["trial_id"].nunique()),
        "n_train_trials": int(train_df["trial_id"].nunique()),
        "n_test_trials": int(test_df["trial_id"].nunique()),
        "class_values": class_values,
        "window_level_classification": window_cls_metrics,
        "window_level_numeric_error": window_num_metrics,
        "trial_level_classification": trial_cls_metrics,
        "trial_level_numeric_error": trial_num_metrics,
    }

    # save
    test_df.to_csv(out_dir / "test_window_predictions.csv", index=False)
    trial_df.to_csv(out_dir / "test_trial_predictions.csv", index=False)
    train_trial_counts_per_weight.to_csv(out_dir / "train_trial_counts_per_weight.csv", index=False)
    test_trial_counts_per_weight.to_csv(out_dir / "test_trial_counts_per_weight.csv", index=False)
    feature_importance_df.to_csv(out_dir / "feature_importance.csv", index=False)
    cm_window_df.to_csv(out_dir / "window_confusion_matrix.csv")
    cm_trial_df.to_csv(out_dir / "trial_confusion_matrix.csv")

    with open(out_dir / "summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print("=== RF classifier (no fatigue) complete ===")
    print(f"Split strategy: {split_strategy}")
    print(f"Rows removed by fatigue filter: {n_before - n_after}")
    print(f"Train windows: {len(train_df)} | Test windows: {len(test_df)}")
    print(f"Train trials: {train_df['trial_id'].nunique()} | Test trials: {test_df['trial_id'].nunique()}")
    print()
    print("Window-level classification")
    print(window_cls_metrics)
    print("Window-level numeric error")
    print(window_num_metrics)
    print()
    print("Trial-level classification (main)")
    print(trial_cls_metrics)
    print("Trial-level numeric error")
    print(trial_num_metrics)
    print()
    print("Top 20 feature importances")
    print(feature_importance_df.head(20).to_string(index=False))


if __name__ == "__main__":
    main()`

=== RF classifier (no fatigue) complete ===
Split strategy: subject_level
Rows removed by fatigue filter: 61442
Train windows: 241090 | Test windows: 74686
Train trials: 289 | Test trials: 80

Window-level classification
{'accuracy': 0.27446911067669977, 'balanced_accuracy': 0.2789436549231173, 'macro_f1': 0.25971849186924467}
Window-level numeric error
{'MAE': 3.1402993867659275, 'RMSE': 4.164009024876556}

Trial-level classification (main)
{'accuracy': 0.4625, 'balanced_accuracy': 0.4625, 'macro_f1': 0.4266462219950592}
Trial-level numeric error
{'MAE': 1.8575, 'RMSE': 2.8329313440321844}

Top 20 feature importances
       feature  importance
  EMG_CH5_iemg    0.035539
  EMG_CH5_mean    0.034614
   EMG_CH5_min    0.030245
   EMG_CH5_mav    0.029698
   EMG_CH5_rms    0.028239
EMG_CH5_median    0.028200
  EMG_CH4_iemg    0.023537
   EMG_CH4_min    0.021690
EMG_CH4_median    0.021003
  EMG_CH4_mean    0.020529
   EMG_CH4_mav    0.020059
   EMG_CH2_mav    0.019751
   EMG_CH4_rms    0.018

In [18]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =========================
# PATHS
# =========================
INPUT_PARQUET = r"C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\EMG\TimeFreq_200_100\Merged_Features\Merged_Features_TimeFreq_200_100.parquet"
INPUT_CSV = r"C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\EMG\TimeFreq_200_100\Merged_Features\Merged_Features_TimeFreq_200_100.csv"
OUTPUT_DIR = r"C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\EMG\TimeFreq_200_100\Results"

VIS_DIR = Path(OUTPUT_DIR) / "Visualizations"
VIS_DIR.mkdir(parents=True, exist_ok=True)

TRIAL_PRED_CSV = Path(OUTPUT_DIR) / "test_trial_predictions.csv"
WINDOW_PRED_CSV = Path(OUTPUT_DIR) / "test_window_predictions.csv"
TRIAL_CM_CSV = Path(OUTPUT_DIR) / "trial_confusion_matrix.csv"
WINDOW_CM_CSV = Path(OUTPUT_DIR) / "window_confusion_matrix.csv"


# =========================
# HELPERS
# =========================
def normalize_sex(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s in {"m", "male"}:
        return "Male"
    if s in {"f", "female"}:
        return "Female"
    return str(x)

def plot_confusion_matrix_from_csv(csv_path, out_path, title):
    cm_df = pd.read_csv(csv_path, index_col=0)

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm_df.values, aspect="auto")

    ax.set_xticks(np.arange(cm_df.shape[1]))
    ax.set_yticks(np.arange(cm_df.shape[0]))
    ax.set_xticklabels(cm_df.columns, rotation=45, ha="right")
    ax.set_yticklabels(cm_df.index)

    for i in range(cm_df.shape[0]):
        for j in range(cm_df.shape[1]):
            ax.text(j, i, str(cm_df.iloc[i, j]), ha="center", va="center", fontsize=9)

    ax.set_title(title)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()

def find_true_pred_cols(df):
    true_candidates = ["y_true_weight", "y_true_trial", "Box Mass", "Box Weight"]
    pred_candidates = ["y_pred_weight", "y_pred_trial"]

    true_col = None
    pred_col = None

    for c in true_candidates:
        if c in df.columns:
            true_col = c
            break

    for c in pred_candidates:
        if c in df.columns:
            pred_col = c
            break

    if true_col is None or pred_col is None:
        raise ValueError(f"Could not find true/pred columns. Columns: {df.columns.tolist()}")

    return true_col, pred_col


# =========================
# LOAD
# =========================
trial_df = pd.read_csv(TRIAL_PRED_CSV)
window_df = pd.read_csv(WINDOW_PRED_CSV)

trial_true_col, trial_pred_col = find_true_pred_cols(trial_df)
window_true_col, window_pred_col = find_true_pred_cols(window_df)

if "Sex" in trial_df.columns:
    trial_df["Sex"] = trial_df["Sex"].map(normalize_sex)

trial_df["error"] = trial_df[trial_pred_col] - trial_df[trial_true_col]
trial_df["abs_error"] = np.abs(trial_df["error"])


# =========================
# 1. trial-level scatter: true vs pred
# =========================
plt.figure(figsize=(6, 6))
plt.scatter(trial_df[trial_true_col], trial_df[trial_pred_col], alpha=0.7)
min_v = min(trial_df[trial_true_col].min(), trial_df[trial_pred_col].min())
max_v = max(trial_df[trial_true_col].max(), trial_df[trial_pred_col].max())
plt.plot([min_v, max_v], [min_v, max_v], linewidth=1)
plt.xlabel("True weight")
plt.ylabel("Predicted weight")
plt.title("Trial-level prediction vs true")
plt.tight_layout()
plt.savefig(VIS_DIR / "trial_scatter_true_vs_pred.png", dpi=300, bbox_inches="tight")
plt.close()


# =========================
# 2. trial-level absolute error by true weight: boxplot
# =========================
weight_levels = sorted(pd.to_numeric(trial_df[trial_true_col], errors="coerce").dropna().unique().tolist())
box_data = [trial_df.loc[trial_df[trial_true_col] == w, "abs_error"].values for w in weight_levels]

plt.figure(figsize=(8, 5))
plt.boxplot(box_data, labels=[str(w) for w in weight_levels])
plt.xlabel("True weight")
plt.ylabel("Absolute error")
plt.title("Trial-level absolute error by weight")
plt.tight_layout()
plt.savefig(VIS_DIR / "trial_abs_error_by_weight_boxplot.png", dpi=300, bbox_inches="tight")
plt.close()


# =========================
# 3. mean abs error by true weight
# =========================
per_weight_df = (
    trial_df.groupby(trial_true_col, as_index=False)
    .agg(
        n_trials=("abs_error", "size"),
        mean_pred=(trial_pred_col, "mean"),
        mean_error=("error", "mean"),
        mean_abs_error=("abs_error", "mean"),
        median_abs_error=("abs_error", "median"),
    )
    .sort_values(trial_true_col)
    .reset_index(drop=True)
)

per_weight_df.to_csv(VIS_DIR / "per_weight_error_summary.csv", index=False)

plt.figure(figsize=(8, 5))
plt.bar(per_weight_df[trial_true_col].astype(str), per_weight_df["mean_abs_error"])
plt.xlabel("True weight")
plt.ylabel("Mean absolute error")
plt.title("Trial-level mean absolute error by weight")
plt.tight_layout()
plt.savefig(VIS_DIR / "trial_mean_abs_error_by_weight_bar.png", dpi=300, bbox_inches="tight")
plt.close()


# =========================
# 4. signed error by true weight
# =========================
plt.figure(figsize=(8, 5))
plt.bar(per_weight_df[trial_true_col].astype(str), per_weight_df["mean_error"])
plt.axhline(0, linewidth=1)
plt.xlabel("True weight")
plt.ylabel("Mean signed error")
plt.title("Trial-level signed error by weight")
plt.tight_layout()
plt.savefig(VIS_DIR / "trial_mean_signed_error_by_weight_bar.png", dpi=300, bbox_inches="tight")
plt.close()


# =========================
# 5. per-sex trial-level mean abs error
# =========================
if "Sex" in trial_df.columns:
    per_sex_df = (
        trial_df.groupby("Sex", as_index=False)
        .agg(
            n_trials=("abs_error", "size"),
            mean_abs_error=("abs_error", "mean"),
            median_abs_error=("abs_error", "median"),
        )
        .sort_values("Sex")
        .reset_index(drop=True)
    )
    per_sex_df.to_csv(VIS_DIR / "per_sex_error_summary.csv", index=False)

    plt.figure(figsize=(6, 4))
    plt.bar(per_sex_df["Sex"], per_sex_df["mean_abs_error"])
    plt.xlabel("Sex")
    plt.ylabel("Mean absolute error")
    plt.title("Trial-level mean absolute error by sex")
    plt.tight_layout()
    plt.savefig(VIS_DIR / "trial_mean_abs_error_by_sex_bar.png", dpi=300, bbox_inches="tight")
    plt.close()


# =========================
# 6. rounded confusion-style table from trial predictions
# =========================
trial_df["pred_weight_rounded"] = pd.to_numeric(trial_df[trial_pred_col], errors="coerce").round().astype(int)
rounded_confusion = pd.crosstab(trial_df[trial_true_col], trial_df["pred_weight_rounded"])
rounded_confusion.to_csv(VIS_DIR / "rounded_weight_confusion_style.csv")


# =========================
# 7. saved model confusion matrices
# =========================
if TRIAL_CM_CSV.exists():
    plot_confusion_matrix_from_csv(
        TRIAL_CM_CSV,
        VIS_DIR / "trial_confusion_matrix.png",
        "Trial-level confusion matrix"
    )

if WINDOW_CM_CSV.exists():
    plot_confusion_matrix_from_csv(
        WINDOW_CM_CSV,
        VIS_DIR / "window_confusion_matrix.png",
        "Window-level confusion matrix"
    )


# =========================
# 8. quick summary print
# =========================
print("Saved visualizations to:")
print(VIS_DIR)

print("\nPer-weight summary:")
print(per_weight_df)

if "Sex" in trial_df.columns:
    print("\nPer-sex summary:")
    print(per_sex_df)

Saved visualizations to:
C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\EMG\TimeFreq_200_100\Results\Visualizations

Per-weight summary:
   y_true_weight  n_trials  mean_pred  mean_error  mean_abs_error  \
0            2.3        16    2.71250     0.41250         0.41250   
1            4.5        16    5.52500     1.02500         2.40000   
2            6.8        16    7.50625     0.70625         2.40625   
3            9.1        16    8.76875    -0.33125         2.80625   
4           11.3        16   10.03750    -1.26250         1.26250   

   median_abs_error  
0               0.0  
1               2.2  
2               2.3  
3               2.2  
4               0.0  

Per-sex summary:
      Sex  n_trials  mean_abs_error  median_abs_error
0  Female        40          1.7975               2.2
1    Male        40          1.9175               2.2


In [9]:
# emg_rf_classifier_no_fatigue_tuned.py

import json
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
)
from sklearn.model_selection import GroupKFold, GroupShuffleSplit


# =========================
# CONFIG
# =========================
INPUT_PARQUET = r"C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\EMG\TimeFreq_200_100\Merged_Features\Merged_Features_TimeFreq_200_100.parquet"
OUTPUT_DIR = r"C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\EMG\TimeFreq_200_100\Results_Tuned"

RANDOM_STATE = 42
TEST_SIZE = 0.20
N_INNER_SPLITS = 3

PARTICIPANT_CANDIDATES = ["Participant"]
SEX_CANDIDATES = ["Sex"]
TRIAL_NAME_CANDIDATES = ["Trial Number"]
TRIAL_ID_CANDIDATES = ["Timeline", "Timestamp", "Trial Number"]
TARGET_CANDIDATES = ["Box Weight", "Box Mass"]

# PARAM_GRID = {
#     "n_estimators": [200, 300, 500],
#     "max_depth": [15, 25, None],
#     "min_samples_leaf": [1, 3, 5],
#     "max_features": ["sqrt"],
#     "class_weight": ["balanced_subsample"],
# }

# PARAM_GRID = {
#     "n_estimators": [200],
#     "max_depth": [5, 8, 10, 12, 15],
#     "min_samples_leaf": [3, 5],
#     "max_features": ["sqrt"],
#     "class_weight": ["balanced_subsample"],
# }

PARAM_GRID = {
    "n_estimators": [300],
    "max_depth": [5, 8, 10, 12],
    "min_samples_leaf": [3],
    "max_features": ["sqrt"],
    "class_weight": ["balanced_subsample"],
}

# =========================
# HELPERS
# =========================
def ensure_dir(path):
    p = Path(path)
    p.mkdir(parents=True, exist_ok=True)
    return p


def resolve_column(df, candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise ValueError(f"None of these columns found: {candidates}")
    return ""


def normalize_sex(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s in {"m", "male"}:
        return "Male"
    if s in {"f", "female"}:
        return "Female"
    return str(x)


def build_trial_id(df, participant_col, trial_id_col):
    return (
        df[participant_col].astype(str).fillna("NA")
        + "||"
        + df[trial_id_col].astype(str).fillna("NA")
    )


def select_feature_columns(df, participant_col, sex_col, trial_id_col, target_col):
    excluded = {
        participant_col,
        sex_col,
        trial_id_col,
        target_col,
        "trial_id",
        "source_file",
        "start_time",
        "end_time",
        "window_start_idx",
        "window_end_idx",
        "window_size",
        "step_size",
        "Activity",
        "Load Type",
        "Lifting Height",
        "Lifting Depth",
        "Lowering Height",
        "Trial Number",
        "Age",
        "Stature",
        "Body Mass",
        "Timestamp",
        "Timeline",
        "Box Weight",
        "Box Mass",
    }

    feature_cols = [c for c in df.columns if c.startswith("EMG_CH") and c not in excluded]
    if not feature_cols:
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        feature_cols = [c for c in numeric_cols if c not in excluded]

    if not feature_cols:
        raise ValueError("No usable feature columns found.")

    return sorted(feature_cols)


def classification_metrics(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
    }


def numeric_error_metrics(y_true_weight, y_pred_weight):
    y_true_weight = np.asarray(y_true_weight, dtype=float)
    y_pred_weight = np.asarray(y_pred_weight, dtype=float)
    return {
        "MAE": float(mean_absolute_error(y_true_weight, y_pred_weight)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true_weight, y_pred_weight))),
    }


def aggregate_trial_probs(df_eval, prob_cols, participant_col, sex_col, trial_id_col, target_col):
    agg_prob = df_eval.groupby("trial_id", as_index=False)[prob_cols].mean()

    meta = (
        df_eval.groupby("trial_id", as_index=False)
        .agg({
            participant_col: "first",
            sex_col: "first",
            trial_id_col: "first",
            target_col: "first",
        })
    )

    return meta.merge(agg_prob, on="trial_id", how="left")


def evaluate_classifier_on_df(
    clf,
    eval_df,
    feature_cols,
    participant_col,
    sex_col,
    trial_id_col,
    target_col,
    idx_to_class,
    class_to_idx,
):
    eval_df = eval_df.copy()

    X = eval_df[feature_cols].values
    eval_df["y_true_class"] = eval_df[target_col].map(class_to_idx).astype(int)
    eval_df["y_pred_class"] = clf.predict(X)
    eval_df["y_true_weight"] = eval_df["y_true_class"].map(idx_to_class)
    eval_df["y_pred_weight"] = eval_df["y_pred_class"].map(idx_to_class)

    prob = clf.predict_proba(X)
    prob_cols = [f"prob_{idx_to_class[i]}" for i in range(len(idx_to_class))]
    prob_df = pd.DataFrame(prob, columns=prob_cols, index=eval_df.index)
    eval_df = pd.concat([eval_df, prob_df], axis=1)

    window_cls = classification_metrics(eval_df["y_true_class"], eval_df["y_pred_class"])
    window_num = numeric_error_metrics(eval_df["y_true_weight"], eval_df["y_pred_weight"])

    trial_df = aggregate_trial_probs(
        df_eval=eval_df,
        prob_cols=prob_cols,
        participant_col=participant_col,
        sex_col=sex_col,
        trial_id_col=trial_id_col,
        target_col=target_col,
    )

    prob_mat = trial_df[prob_cols].values
    trial_df["y_pred_class"] = np.argmax(prob_mat, axis=1)
    trial_df["y_pred_weight"] = trial_df["y_pred_class"].map(idx_to_class)
    trial_df["y_true_weight"] = pd.to_numeric(trial_df[target_col], errors="coerce")
    trial_df["y_true_class"] = trial_df["y_true_weight"].map(class_to_idx).astype(int)

    trial_cls = classification_metrics(trial_df["y_true_class"], trial_df["y_pred_class"])
    trial_num = numeric_error_metrics(trial_df["y_true_weight"], trial_df["y_pred_weight"])

    return eval_df, trial_df, window_cls, window_num, trial_cls, trial_num


def param_grid_to_list(param_grid):
    keys = list(param_grid.keys())
    vals = [param_grid[k] for k in keys]
    return [dict(zip(keys, v)) for v in itertools.product(*vals)]


# =========================
# MAIN
# =========================
def main():
    out_dir = ensure_dir(OUTPUT_DIR)

    df = pd.read_parquet(INPUT_PARQUET)

    participant_col = resolve_column(df, PARTICIPANT_CANDIDATES, required=True)
    sex_col = resolve_column(df, SEX_CANDIDATES, required=True)
    trial_name_col = resolve_column(df, TRIAL_NAME_CANDIDATES, required=False)
    trial_id_col = resolve_column(df, TRIAL_ID_CANDIDATES, required=True)
    target_col = resolve_column(df, TARGET_CANDIDATES, required=True)

    df[sex_col] = df[sex_col].map(normalize_sex)

    # remove fatigue
    n_before = len(df)
    if trial_name_col:
        fatigue_mask = df[trial_name_col].astype(str).str.contains("fatigue", case=False, na=False)
        df = df.loc[~fatigue_mask].copy()
    n_after = len(df)

    df["trial_id"] = build_trial_id(df, participant_col, trial_id_col)

    feature_cols = select_feature_columns(df, participant_col, sex_col, trial_id_col, target_col)

    keep_cols = [participant_col, sex_col, trial_id_col, target_col, "trial_id"] + feature_cols
    optional_cols = [
        c for c in [
            "Trial Number",
            "Activity",
            "Lifting Height",
            "Lowering Height",
            "Load Type",
            "Lifting Depth",
            "Age",
            "Stature",
            "Body Mass",
            "window_start_idx",
            "window_end_idx",
            "window_size",
            "step_size",
            "source_file",
        ]
        if c in df.columns and c not in keep_cols
    ]
    work_df = df[keep_cols + optional_cols].copy()

    work_df = work_df.dropna(subset=[participant_col, sex_col, trial_id_col, target_col]).copy()
    X_all = work_df[feature_cols].replace([np.inf, -np.inf], np.nan)
    valid_mask = X_all.notna().all(axis=1)
    work_df = work_df.loc[valid_mask].reset_index(drop=True)

    if len(work_df) == 0:
        raise ValueError("No valid rows remain after filtering.")

    class_values = sorted(pd.to_numeric(work_df[target_col], errors="coerce").dropna().unique().tolist())
    class_to_idx = {v: i for i, v in enumerate(class_values)}
    idx_to_class = {i: v for v, i in class_to_idx.items()}
    work_df["y_class"] = work_df[target_col].map(class_to_idx).astype(int)

    # outer subject-level split
    outer_groups = work_df[participant_col].astype(str).values
    gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    train_idx, test_idx = next(gss.split(work_df, groups=outer_groups))

    train_df = work_df.iloc[train_idx].reset_index(drop=True)
    test_df = work_df.iloc[test_idx].reset_index(drop=True)

    shared_trials = set(train_df["trial_id"]).intersection(set(test_df["trial_id"]))
    if shared_trials:
        raise RuntimeError("Leakage detected between train and test.")

    # inner grouped CV tuning on train subjects only
    unique_train_subjects = train_df[participant_col].astype(str).nunique()
    n_splits = min(N_INNER_SPLITS, unique_train_subjects)
    if n_splits < 2:
        raise ValueError("Not enough training subjects for grouped CV tuning.")

    inner_cv = GroupKFold(n_splits=n_splits)
    param_list = param_grid_to_list(PARAM_GRID)
    cv_rows = []

    print(f"Total parameter combinations: {len(param_list)}")
    print(f"Inner CV splits: {n_splits}")

    for i, params in enumerate(param_list, 1):
        fold_scores = []

        for fold_id, (tr_idx, va_idx) in enumerate(
            inner_cv.split(train_df, groups=train_df[participant_col].astype(str).values), 1
        ):
            tr_df = train_df.iloc[tr_idx].reset_index(drop=True)
            va_df = train_df.iloc[va_idx].reset_index(drop=True)

            clf = RandomForestClassifier(
                random_state=RANDOM_STATE,
                n_jobs=-1,
                bootstrap=True,
                min_samples_split=2,
                **params,
            )
            clf.fit(tr_df[feature_cols].values, tr_df["y_class"].values)

            _, va_trial_df, _, _, va_trial_cls, va_trial_num = evaluate_classifier_on_df(
                clf=clf,
                eval_df=va_df,
                feature_cols=feature_cols,
                participant_col=participant_col,
                sex_col=sex_col,
                trial_id_col=trial_id_col,
                target_col=target_col,
                idx_to_class=idx_to_class,
                class_to_idx=class_to_idx,
            )

            cv_rows.append({
                "param_id": i,
                "fold_id": fold_id,
                **params,
                "trial_balanced_accuracy": va_trial_cls["balanced_accuracy"],
                "trial_accuracy": va_trial_cls["accuracy"],
                "trial_macro_f1": va_trial_cls["macro_f1"],
                "trial_MAE": va_trial_num["MAE"],
                "trial_RMSE": va_trial_num["RMSE"],
                "n_val_trials": int(len(va_trial_df)),
            })
            fold_scores.append(va_trial_cls["balanced_accuracy"])

        print(f"[{i}/{len(param_list)}] mean CV balanced accuracy = {np.mean(fold_scores):.4f} | params = {params}")

    cv_df = pd.DataFrame(cv_rows)

    cv_summary_df = (
        cv_df.groupby(
            ["param_id", "n_estimators", "max_depth", "min_samples_leaf", "max_features", "class_weight"],
            as_index=False
        )
        .agg(
            mean_trial_balanced_accuracy=("trial_balanced_accuracy", "mean"),
            std_trial_balanced_accuracy=("trial_balanced_accuracy", "std"),
            mean_trial_accuracy=("trial_accuracy", "mean"),
            mean_trial_macro_f1=("trial_macro_f1", "mean"),
            mean_trial_MAE=("trial_MAE", "mean"),
            mean_trial_RMSE=("trial_RMSE", "mean"),
        )
        .sort_values(
            ["mean_trial_balanced_accuracy", "mean_trial_macro_f1", "mean_trial_accuracy"],
            ascending=[False, False, False]
        )
        .reset_index(drop=True)
    )

    best_row = cv_summary_df.iloc[0]
    best_params = {
        "n_estimators": int(best_row["n_estimators"]),
        "max_depth": None if pd.isna(best_row["max_depth"]) else int(best_row["max_depth"]),
        "min_samples_leaf": int(best_row["min_samples_leaf"]),
        "max_features": best_row["max_features"],
        "class_weight": best_row["class_weight"],
    }

    # fit best model on full train
    best_clf = RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        bootstrap=True,
        min_samples_split=2,
        **best_params,
    )
    best_clf.fit(train_df[feature_cols].values, train_df["y_class"].values)

    test_window_df, test_trial_df, window_cls, window_num, trial_cls, trial_num = evaluate_classifier_on_df(
        clf=best_clf,
        eval_df=test_df,
        feature_cols=feature_cols,
        participant_col=participant_col,
        sex_col=sex_col,
        trial_id_col=trial_id_col,
        target_col=target_col,
        idx_to_class=idx_to_class,
        class_to_idx=class_to_idx,
    )

    cm_window = confusion_matrix(
        test_window_df["y_true_class"],
        test_window_df["y_pred_class"],
        labels=list(range(len(class_values))),
    )
    cm_trial = confusion_matrix(
        test_trial_df["y_true_class"],
        test_trial_df["y_pred_class"],
        labels=list(range(len(class_values))),
    )

    cm_window_df = pd.DataFrame(
        cm_window,
        index=[f"true_{v}" for v in class_values],
        columns=[f"pred_{v}" for v in class_values],
    )
    cm_trial_df = pd.DataFrame(
        cm_trial,
        index=[f"true_{v}" for v in class_values],
        columns=[f"pred_{v}" for v in class_values],
    )

    feature_importance_df = pd.DataFrame({
        "feature": feature_cols,
        "importance": best_clf.feature_importances_,
    }).sort_values("importance", ascending=False).reset_index(drop=True)

    summary = {
        "input_parquet": INPUT_PARQUET,
        "n_rows_before_removing_fatigue": int(n_before),
        "n_rows_after_removing_fatigue": int(n_after),
        "n_removed_rows": int(n_before - n_after),
        "n_total_windows_used": int(len(work_df)),
        "n_train_windows": int(len(train_df)),
        "n_test_windows": int(len(test_df)),
        "n_total_trials_used": int(work_df["trial_id"].nunique()),
        "n_train_trials": int(train_df["trial_id"].nunique()),
        "n_test_trials": int(test_df["trial_id"].nunique()),
        "n_total_subjects": int(work_df[participant_col].nunique()),
        "n_train_subjects": int(train_df[participant_col].nunique()),
        "n_test_subjects": int(test_df[participant_col].nunique()),
        "class_values": class_values,
        "best_params": best_params,
        "window_level_classification": window_cls,
        "window_level_numeric_error": window_num,
        "trial_level_classification": trial_cls,
        "trial_level_numeric_error": trial_num,
    }

    cv_df.to_csv(out_dir / "cv_fold_results.csv", index=False)
    cv_summary_df.to_csv(out_dir / "cv_param_summary.csv", index=False)
    test_window_df.to_csv(out_dir / "test_window_predictions.csv", index=False)
    test_trial_df.to_csv(out_dir / "test_trial_predictions.csv", index=False)
    cm_window_df.to_csv(out_dir / "window_confusion_matrix.csv")
    cm_trial_df.to_csv(out_dir / "trial_confusion_matrix.csv")
    feature_importance_df.to_csv(out_dir / "feature_importance.csv", index=False)

    with open(out_dir / "summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print("\n=== RF classifier tuned (no fatigue) complete ===")
    print("Best params:")
    print(best_params)
    print("\nWindow-level classification")
    print(window_cls)
    print("Window-level numeric error")
    print(window_num)
    print("\nTrial-level classification (main)")
    print(trial_cls)
    print("Trial-level numeric error")
    print(trial_num)
    print("\nTop 10 CV rows")
    print(cv_summary_df.head(10).to_string(index=False))


if __name__ == "__main__":
    main()

Total parameter combinations: 4
Inner CV splits: 3
[1/4] mean CV balanced accuracy = 0.3626 | params = {'n_estimators': 300, 'max_depth': 5, 'min_samples_leaf': 3, 'max_features': 'sqrt', 'class_weight': 'balanced_subsample'}
[2/4] mean CV balanced accuracy = 0.4253 | params = {'n_estimators': 300, 'max_depth': 8, 'min_samples_leaf': 3, 'max_features': 'sqrt', 'class_weight': 'balanced_subsample'}
[3/4] mean CV balanced accuracy = 0.4290 | params = {'n_estimators': 300, 'max_depth': 10, 'min_samples_leaf': 3, 'max_features': 'sqrt', 'class_weight': 'balanced_subsample'}
[4/4] mean CV balanced accuracy = 0.4146 | params = {'n_estimators': 300, 'max_depth': 12, 'min_samples_leaf': 3, 'max_features': 'sqrt', 'class_weight': 'balanced_subsample'}

=== RF classifier tuned (no fatigue) complete ===
Best params:
{'n_estimators': 300, 'max_depth': 10, 'min_samples_leaf': 3, 'max_features': 'sqrt', 'class_weight': 'balanced_subsample'}

Window-level classification
{'accuracy': 0.274469110676699